Setup and load dataset

In [8]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import nltk

nltk.download("punkt")
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

from google.colab import drive
drive.mount('/content/drive')

BASE = Path("/content/drive/MyDrive/CS685")
DATA_DIR = BASE / "linkedin"

POSTINGS_PATH = DATA_DIR / "postings.csv"

jobs = pd.read_csv(POSTINGS_PATH)
print(jobs.shape)
jobs.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(123849, 31)


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,This position requires a baseline understandin...,1.712896e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,157500.0,11040.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,1.713452e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,70000.0,52601.0,19057.0


Merge description and skills_desc

In [9]:
def merge_desc_and_skills(row):
    desc = str(row.get("description", "") or "")
    skills = str(row.get("skills_desc", "") or "")
    desc = desc.strip()
    skills = skills.strip()
    if skills:
        return desc + "\n\nSkills / Extra Info:\n" + skills
    else:
        return desc

jobs["full_text"] = jobs.apply(merge_desc_and_skills, axis=1)
jobs = jobs[jobs["full_text"].str.strip() != ""].reset_index(drop=True)
print("After dropping empty descriptions:", jobs.shape)

After dropping empty descriptions: (123849, 32)


Infer domain from title (DS / SWE / DA / OTHER)

In [10]:
def infer_domain(title):
    t = str(title).lower()

    # Data Science / ML
    ds_keywords = [
        "data scientist", "data science", "machine learning", "ml engineer",
        "ml scientist", "research scientist", "applied scientist", "ai engineer",
        "ai scientist"
    ]
    # Software / SWE
    swe_keywords = [
        "software engineer", "software developer", "backend engineer",
        "front end engineer", "frontend engineer", "full stack", "full-stack",
        "sde", "swe", "developer", "devops engineer"
    ]
    # Analyst
    da_keywords = [
        "data analyst", "business analyst", "analytics engineer",
        "analytics", "bi analyst", "business intelligence", "insights analyst"
    ]

    for kw in ds_keywords:
        if kw in t:
            return "DS"
    for kw in swe_keywords:
        if kw in t:
            return "SWE"
    for kw in da_keywords:
        if kw in t:
            return "DA"
    return "OTHER"

jobs["domain"] = jobs["title"].apply(infer_domain)

jobs["domain"].value_counts()

,count
domain,
OTHER,117442
SWE,4415
DA,1395
DS,597


In [11]:
jobs = jobs[jobs["domain"].isin(["DS", "SWE", "DA"])].reset_index(drop=True)
print("After domain filter:", jobs.shape)

After domain filter: (6407, 33)


Clean text

In [12]:
def clean_text(t):
    t = str(t)
    # strip HTML tags
    t = re.sub(r"<.*?>", " ", t)
    # weird bullets etc. – optional
    t = t.replace("\xa0", " ")
    # collapse whitespace
    t = re.sub(r"\s+", " ", t)
    return t.strip()

jobs["cleaned_text"] = jobs["full_text"].apply(clean_text)

In [13]:
jobs_small = jobs[["job_id", "company_name", "title", "domain", "cleaned_text"]].copy()
JOBS_OUT = DATA_DIR / "jobs_cleaned.csv"
jobs_small.to_csv(JOBS_OUT, index=False)
JOBS_OUT

PosixPath('/content/drive/MyDrive/CS685/linkedin/jobs_cleaned.csv')

Split into sentences

In [14]:
rows = []

for _, row in jobs_small.iterrows():
    job_id = row["job_id"]
    domain = row["domain"]
    text = row["cleaned_text"]

    for s_idx, sent in enumerate(sent_tokenize(text)):
        sent = sent.strip()
        if not sent:
            continue
        rows.append({
            "job_id": job_id,
            "domain": domain,
            "sent_idx": s_idx,
            "sentence": sent,
        })

sent_df = pd.DataFrame(rows)
print("Total sentences:", len(sent_df))
sent_df.head()

Total sentences: 100770


,job_id,domain,sent_idx,sentence
0,133130219,SWE,0,"Education Bachelor's degree in software, math,..."
1,175485704,SWE,0,Job Description:GOYT is seeking a skilled and ...
2,175485704,SWE,1,"As a key member of our development team, you w..."
3,175485704,SWE,2,This is an equity-based role.
4,175485704,SWE,3,Responsibilities:Develop and maintain high-qua...


Flag “likely skill” sentences

In [15]:
def likely_skill_sentence(s):
    tokens = s.split()
    if len(tokens) < 5 or len(tokens) > 60:
        return False

    s_lower = s.lower()
    keywords = [
        "experience with",
        "experience in",
        "experience using",
        "knowledge of",
        "skills in",
        "skills with",
        "proficient in",
        "familiarity with",
        "background in",
        "expertise in",
        "responsible for",
        "will be responsible for",
        "ability to",
        "required skills",
        "requirements:",
        "what you will do",
        "you will",
        "you'll",
        "we are looking for",
        "we’re looking for",
        "qualifications",
        "preferred qualifications",
        "nice to have",
        "must have",
    ]
    return any(k in s_lower for k in keywords)

sent_df["likely_skill"] = sent_df["sentence"].apply(likely_skill_sentence)
filtered = sent_df[sent_df["likely_skill"]].copy()
print("Likely skill sentences:", len(filtered))

Likely skill sentences: 15639


In [16]:
filtered = filtered.drop_duplicates(subset=["sentence"]).reset_index(drop=True)
print("After dedup:", len(filtered))

After dedup: 12589


Sample sentences for annotation

In [18]:
N_SAMPLES = 1000

if len(filtered) <= N_SAMPLES:
    sample_df = filtered.copy()
    print(f"Only {len(filtered)} sentences; using all of them.")
else:
    sample_df = filtered.sample(n=N_SAMPLES, random_state=42).reset_index(drop=True)

# Create sent_id + empty annotation columns
sample_df["sent_id"] = sample_df["job_id"].astype(str) + "_" + sample_df["sent_idx"].astype(str)
sample_df["has_skill"] = ""  # optional
sample_df["spans"] = ""      # to be filled during annotation

# reorder
sample_df = sample_df[["job_id", "domain", "sent_id", "sentence", "has_skill", "spans"]]

ANNOT_PATH = DATA_DIR / "sentences_for_annotation_v2.csv"
sample_df.to_csv(ANNOT_PATH, index=False)
ANNOT_PATH

PosixPath('/content/drive/MyDrive/CS685/linkedin/sentences_for_annotation_v2.csv')